In [1]:
# 라이브러리 import

import os
import shutil
import subprocess
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink

In [2]:
# Ghostscript 압축 함수

def compress_pdf_ghostscript(input_path, output_path, pdf_setting="/ebook"):
    """
    Ghostscript를 사용해 PDF를 압축합니다.

    Parameters
    ----------
    input_path : str | Path
        원본 PDF 경로
    output_path : str | Path
        저장할 PDF 경로
    pdf_setting : str
        /screen   : 가장 강한 압축, 화질 낮음
        /ebook    : 균형형
        /printer  : 화질 우선
        /prepress : 고화질
    """
    gs_executable = shutil.which("gs")
    if gs_executable is None:
        raise RuntimeError(
            "Ghostscript(gs)를 찾을 수 없습니다. 터미널에서 `brew install ghostscript` 후 다시 시도하세요."
        )

    cmd = [
        gs_executable,
        "-sDEVICE=pdfwrite",
        "-dCompatibilityLevel=1.4",
        f"-dPDFSETTINGS={pdf_setting}",
        "-dNOPAUSE",
        "-dQUIET",
        "-dBATCH",
        f"-sOutputFile={str(output_path)}",
        str(input_path),
    ]

    subprocess.run(cmd, check=True)

In [3]:
# 드래그 앤 드롭 UI

upload = widgets.FileUpload(
    accept=".pdf",
    multiple=False,
    description="PDF 업로드"
)

setting_dropdown = widgets.Dropdown(
    options=[
        ("용량 최대 절감 (/screen)", "/screen"),
        ("균형형 (/ebook)", "/ebook"),
        ("화질 우선 (/printer)", "/printer"),
        ("고화질 (/prepress)", "/prepress"),
    ],
    value="/ebook",
    description="압축옵션"
)

run_button = widgets.Button(
    description="압축 실행",
    button_style="success"
)

output = widgets.Output()

In [ ]:
# 실행 로직

def get_uploaded_file(upload_widget):
    """
    ipywidgets 버전 차이를 고려해 업로드 파일 정보를 꺼냅니다.
    반환: (file_name, content_bytes)
    """
    value = upload_widget.value

    if not value:
        return None, None

    if isinstance(value, dict):
        uploaded = list(value.values())[0]
        return uploaded["name"], uploaded["content"]

    if isinstance(value, tuple):
        uploaded = value[0]
        return uploaded["name"], uploaded["content"]

    raise TypeError(f"지원하지 않는 upload.value 타입: {type(value)}")


def on_run_clicked(b):
    with output:
        clear_output()

        try:
            file_name, content = get_uploaded_file(upload)

            if file_name is None:
                print("먼저 PDF 파일을 드래그 앤 드롭하거나 업로드하세요.")
                return

            input_path = Path(file_name).name
            stem = Path(file_name).stem
            suffix = Path(file_name).suffix or ".pdf"
            output_path = f"{stem}_compressed{suffix}"

            # 업로드된 원본 파일 저장
            with open(input_path, "wb") as f:
                f.write(content)

            print(f"원본 저장: {input_path}")
            print(f"압축 옵션: {setting_dropdown.value}")

            compress_pdf_ghostscript(
                input_path=input_path,
                output_path=output_path,
                pdf_setting=setting_dropdown.value
            )

            original_size = os.path.getsize(input_path) / 1024 / 1024
            compressed_size = os.path.getsize(output_path) / 1024 / 1024
            reduction = 0.0

            if original_size > 0:
                reduction = (1 - compressed_size / original_size) * 100

            print(f"압축 완료: {output_path}")
            print(f"원본 크기: {original_size:.2f} MB")
            print(f"압축 크기: {compressed_size:.2f} MB")
            print(f"감소율: {reduction:.1f}%")

            display(FileLink(output_path, result_html_prefix="다운로드: "))

        except subprocess.CalledProcessError as e:
            print("Ghostscript 실행 중 오류가 발생했습니다.")
            print(e)

        except Exception as e:
            print("오류가 발생했습니다.")
            print(e)


run_button.on_click(on_run_clicked)

In [ ]:
# UI 표시

display(
    widgets.VBox([
        widgets.HTML("<h3>PDF 압축기 (드래그 앤 드롭 + Ghostscript)</h3>"),
        widgets.HTML("<p>아래 박스에 PDF 파일을 끌어다 놓고, 압축 옵션을 고른 뒤 실행하세요.</p>"),
        upload,
        setting_dropdown,
        run_button,
        output
    ])
)